In [ ]:
## BANKING CUSTOMER CHURN EDA

In [1]:
# Setup and Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Set style once
plt.style.use('default')
sns.set_palette("Set2")

In [3]:
# Load data
df = pd.read_csv(r"C:\Users\mohda\OneDrive\Documents\data analytics important notes\Bank+Customer+Churn\joined_bank2.csv")

In [4]:
# Basic data exploration
print("=== DATA OVERVIEW ===")
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"\nChurn rate: {df['Exited'].mean():.2%}")
print("\n" + "="*50)

=== DATA OVERVIEW ===
   CustomerId   Surname  CreditScore Geography  Gender  Age  Tenure  \
0    15634602  Hargrave          619    France  Female   42       2   
1    15634602  Hargrave          619    France  Female   42       2   
2    15647311      Hill          608     Spain  Female   41       1   
3    15619304      Onio          502    France  Female   42       8   
4    15701354      Boni          699    France  Female   39       1   

   EstimatedSalary    Balance  NumOfProducts  HasCrCard  Tenure.1  \
0        101348.88       0.00              1          1         2   
1        101348.88       0.00              1          1         2   
2        112542.58   83807.86              1          1         1   
3        113931.57  159660.80              3          0         8   
4         93826.63       0.00              2          0         1   

   IsActiveMember  Exited  
0               1       1  
1               1       1  
2               1       0  
3               0       

In [5]:
# Data preprocessing
df['Balance'] = df['Balance'].astype(float)

In [6]:
# Create age and balance groups
df['AgeGroup'] = pd.cut(df['Age'], bins=[18,30,40,50,60,100], 
                       labels=['18-30','31-40','41-50','51-60','60+'])

In [7]:
df['BalanceGroup'] = pd.qcut(df.loc[df['Balance']>0, 'Balance'], 4, 
                            labels=['Q1 (Low)','Q2','Q3','Q4 (High)'])
df['BalanceGroup'] = df['BalanceGroup'].astype(str)
df.loc[df['Balance']==0, 'BalanceGroup'] = 'Zero Balance'

In [26]:
# === VISUALIZATIONS ===
def save_plot(filename, dpi=300):
    # Helper function to save plots consistently
    plt.tight_layout()
    plt.savefig(filename, dpi=dpi, bbox_inches='tight')
    plt.close()

In [27]:
# 1. Age distribution
plt.figure(figsize=(10,6))
sns.histplot(df['Age'], bins="fd", kde=True)
plt.title("Age Distribution", fontsize=14)
plt.xlabel("Age")
plt.ylabel("Count")
save_plot("age_distribution.png")

In [29]:
# 2. Churn by demographics (combined subplot)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gender vs Churn
sns.countplot(x='Gender', hue='Exited', data=df, ax=axes[0])
axes[0].set_title("Churn by Gender")

# Geography vs Churn  
sns.countplot(x='Geography', hue='Exited', data=df, ax=axes[1])
axes[1].set_title("Churn by Country")

# Active Member vs Churn
sns.countplot(x='IsActiveMember', hue='Exited', data=df, ax=axes[2])
axes[2].set_title("Churn by Active Status")

save_plot("demographic_churn_analysis.png")
    

In [30]:
# 3. Churn rates by segments
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Age groups
age_churn = df.groupby('AgeGroup')['Exited'].mean()
age_churn.plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Churn Rate by Age Group')
axes[0,0].tick_params(axis='x', rotation=45)

# Balance groups  
balance_churn = df.groupby('BalanceGroup')['Exited'].mean()
balance_churn.plot(kind='bar', ax=axes[0,1], color='lightcoral')
axes[0,1].set_title('Churn Rate by Balance Group')
axes[0,1].tick_params(axis='x', rotation=45)

# Geography
geo_churn = df.groupby('Geography')['Exited'].mean()
geo_churn.plot(kind='bar', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Churn Rate by Country')

# Gender
gender_churn = df.groupby('Gender')['Exited'].mean()
gender_churn.plot(kind='bar', ax=axes[1,1], color='gold')
axes[1,1].set_title('Churn Rate by Gender')

save_plot("churn_rates_by_segments.png")

C:\Users\mohda\AppData\Local\Temp\ipykernel_17316\797124179.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_churn = df.groupby('AgeGroup')['Exited'].mean()


In [32]:
# === SUMMARY STATISTICS ===
print("\n=== CHURN ANALYSIS SUMMARY ===")

# Numerical features by churn status
print("\nNumerical Features by Churn Status:")
num_cols = ["CreditScore", "Age", "Balance", "EstimatedSalary", "Tenure", "NumOfProducts"]
num_summary = df.groupby("Exited")[num_cols].agg(["mean", "median", "std"])
print(num_summary.round(2))

# Categorical features churn rates
print("\nChurn Rates by Category:")
categorical_cols = ['Gender', 'Geography', 'IsActiveMember', 'HasCrCard']
for col in categorical_cols:
    churn_rate = df.groupby(col)['Exited'].mean()
    print(f"\n{col}:")
    for category, rate in churn_rate.items():
        print(f"  {category}: {rate:.2%}")

# Key insights
print("\n=== KEY INSIGHTS ===")
print(f"• Highest churn age group: {age_churn.idxmax()} ({age_churn.max():.2%})")
print(f"• Highest churn country: {geo_churn.idxmax()} ({geo_churn.max():.2%})")
print(f"• Highest churn balance group: {balance_churn.idxmax()} ({balance_churn.max():.2%})")
print(f"• Active vs Inactive churn: {df.groupby('IsActiveMember')['Exited'].mean().to_dict()}")


=== CHURN ANALYSIS SUMMARY ===

Numerical Features by Churn Status:
       CreditScore                   Age                 Balance             \
              mean median     std   mean median    std      mean     median   
Exited                                                                        
0           651.93  654.0   95.68  37.40   36.0  10.13  72761.48   92113.61   
1           645.34  646.0  100.30  44.84   45.0   9.76  91063.83  109344.23   

                 EstimatedSalary                      Tenure               \
             std            mean     median       std   mean median   std   
Exited                                                                      
0       62845.67        99717.56   99593.28  57417.59   5.03    5.0  2.88   
1       58381.36       101465.62  102431.88  57898.20   4.93    5.0  2.94   

       NumOfProducts               
                mean median   std  
Exited                             
0               1.54    2.0  0.51  
1    